# LeetCode 2195 — Append K Integers With Minimal Sum bằng Python

Link đề: https://leetcode.com/problems/append-k-integers-with-minimal-sum/ · Độ khó: **Medium** · Chủ đề: Array, Math, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def minimalKSum(self, nums: List[int], k: int) -> int:
        ...
```

Thêm `k` số nguyên dương **khác nhau**, **không có trong `nums`**, sao cho tổng nhỏ nhất ⇒ tham lam: lấy `k` số dương **nhỏ nhất bị thiếu**. Khó khăn: `k` tới `10⁸` nên không thể duyệt từng số. Notebook đi từ **duyệt từng số `1, 2, 3, …`** → **lấp các khoảng trống giữa các phần tử đã sắp bằng công thức cấp số cộng** → **"đẩy ngưỡng" `k` lên mỗi khi gặp số đã có** → **tìm nhị phân ngưỡng + tổng tiền tố** → **bisect trên hàm "số số thiếu ≤ m"**.

## 1. Mục tiêu học

1. Chứng minh tham lam: tập `k` số thiếu có tổng nhỏ nhất là `k` số thiếu **nhỏ nhất**.
2. Tính tổng một đoạn số nguyên liên tiếp `[a, b]` bằng công thức `(a + b)(b − a + 1)/2`.
3. Xử lý **phần tử trùng** trong `nums` (chỉ tính mỗi giá trị một lần).
4. Tìm ngưỡng `m` (số thiếu thứ `k`) bằng "đẩy ngưỡng" hoặc **tìm nhị phân** với hàm `missing(m) = m − #{x ∈ nums, x ≤ m}`.
5. Kiểm thử đối chiếu với duyệt từng số trên dữ liệu nhỏ.

## 2. Đề bài & ví dụ

Cho mảng `nums` và số `k`. Thêm vào `nums` **`k` số nguyên dương khác nhau không xuất hiện trong `nums`** sao cho tổng mới nhỏ nhất. Trả về **tổng của `k` số được thêm**.

**Ví dụ 1:** `nums = [1,4,25,10,25]`, `k = 2` → `5` (thêm `2, 3`).

**Ví dụ 2:** `nums = [5,6]`, `k = 6` → `25` (thêm `1, 2, 3, 4, 7, 8`).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ nums[i] ≤ 10⁹`, `1 ≤ k ≤ 10⁸`. Đáp án tới cỡ `10¹⁶` (cần 64 bit ở C++/Java).

In [1]:
import random
import time
from bisect import bisect_right
from typing import List

examples = [([1, 4, 25, 10, 25], 2, 5), ([5, 6], 6, 25)]

def first_missing(nums, k):
    s = set(nums); out = []; x = 1
    while len(out) < k:
        if x not in s:
            out.append(x)
        x += 1
    return out

for nums, k, exp in examples:
    print(f"nums={nums}, k={k}: {k} số thiếu nhỏ nhất = {first_missing(nums, k)} -> tổng {sum(first_missing(nums, k))} (kỳ vọng {exp})")

nums=[1, 4, 25, 10, 25], k=2: 2 số thiếu nhỏ nhất = [2, 3] -> tổng 5 (kỳ vọng 5)
nums=[5, 6], k=6: 6 số thiếu nhỏ nhất = [1, 2, 3, 4, 7, 8] -> tổng 25 (kỳ vọng 25)


## 3. Kiến thức nền

### 3.1 Tổng đoạn liên tiếp

In [2]:
def range_sum(a, b):
    """Tổng a + (a+1) + ... + b (0 nếu a > b)."""
    return (a + b) * (b - a + 1) // 2 if a <= b else 0

print(range_sum(1, 10), sum(range(1, 11)), "|", range_sum(7, 8), "|", range_sum(5, 4))

55 55 | 15 | 0


### 3.2 Khử trùng + sắp xếp

In [3]:
print(sorted(set([1, 4, 25, 10, 25])))

[1, 4, 10, 25]


### 3.3 `bisect_right` — đếm số phần tử `≤ m` trong mảng đã sắp

In [4]:
a = [1, 4, 10, 25]
for m in [0, 1, 5, 25, 100]:
    print(f"  số phần tử ≤ {m}: {bisect_right(a, m)}")

  số phần tử ≤ 0: 0
  số phần tử ≤ 1: 1
  số phần tử ≤ 5: 2
  số phần tử ≤ 25: 4
  số phần tử ≤ 100: 4


### 3.4 Tổng tiền tố

In [5]:
pre = [0]
for v in a:
    pre.append(pre[-1] + v)
print("tiền tố:", pre, "| tổng 2 phần tử đầu:", pre[2])

tiền tố: [0, 1, 5, 15, 40] | tổng 2 phần tử đầu: 5


## 4. Ý tưởng nền tảng / trực giác

Muốn tổng nhỏ nhất ⇒ chọn các số **nhỏ nhất có thể** chưa có trong `nums`. Tính tay ví dụ 2 (`nums = [5, 6]`, `k = 6`): đi `1, 2, 3, 4` (thiếu — lấy 4 số), `5, 6` (có — bỏ), `7, 8` (lấy 2 số) ⇒ `1+2+3+4+7+8 = 25` ✓.

Nhìn theo **khoảng trống**: các phần tử (đã khử trùng, sắp) chia trục số thành các đoạn trống: `[1, 4]`, `[7, ∞)`. Lấy hết đoạn đầu (4 số), lấy 2 số đầu đoạn sau. Mỗi đoạn tính tổng bằng công thức — không cần duyệt từng số.

Nhìn theo **ngưỡng**: số thiếu thứ `k` là `m`; đáp án `= (1 + … + m) − (tổng các phần tử của nums ≤ m, không trùng)`.

In [6]:
nums, k = [5, 6], 6
u = sorted(set(nums)); prev = 0; left = k
for x in u + [float("inf")]:
    gap_lo, gap_hi = prev + 1, x - 1
    take = min(left, (gap_hi - gap_lo + 1) if x != float("inf") else left)
    if take > 0:
        print(f"  đoạn trống [{gap_lo}, {gap_hi if x != float('inf') else '∞'}]: lấy {take} số {gap_lo}..{gap_lo + take - 1}, tổng {range_sum(gap_lo, gap_lo + take - 1)}")
        left -= take
    if left == 0: break
    prev = x

  đoạn trống [1, 4]: lấy 4 số 1..4, tổng 10
  đoạn trống [7, ∞]: lấy 2 số 7..8, tổng 15


## 5. Cách 1 — Brute Force / baseline: duyệt từng số `1, 2, 3, …`

In [7]:
def min_k_sum_brute(nums, k):
    s = set(nums); total = 0; x = 1
    while k:
        if x not in s:
            total += x; k -= 1
        x += 1
    return total

for nums_, k_, exp in examples:
    print(min_k_sum_brute(nums_, k_), "(kỳ vọng", exp, ")")

5 (kỳ vọng 5 )
25 (kỳ vọng 25 )


**Độ phức tạp:** `O(n + k + n)` — với `k = 10⁸` là 10⁸ vòng Python: quá chậm (hàng chục giây). **Nút thắt:** từng số một trong các đoạn trống dài ⇒ cộng cả đoạn bằng công thức.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề tham lam.** Gọi `g₁ < g₂ < …` là các số dương thiếu. Với mọi tập `k` số thiếu `T = {t₁ < … < t_k}`, ta có `tᵢ ≥ gᵢ` (số thiếu nhỏ thứ `i` của `T` không nhỏ hơn số thiếu nhỏ thứ `i` nói chung) ⇒ `Σ tᵢ ≥ Σ gᵢ`.

**Đẩy ngưỡng.** Bắt đầu `m = k` (nếu không có gì bị chặn, lấy `1..k`). Duyệt các giá trị phân biệt `x` tăng dần: **nếu** `x ≤ m` **thì** `x` chiếm chỗ một số trong `[1, m]` ⇒ phải lấy thêm một số ⇒ `m += 1` (và trừ `x` khỏi tổng). **Ngược lại** dừng (các `x` sau còn lớn hơn). Đáp án `= m(m+1)/2 − Σ_{x ≤ m} x`.

**Tìm nhị phân.** `missing(m) = m − #{x ∈ set(nums) : x ≤ m}` không giảm ⇒ `m` = số nhỏ nhất có `missing(m) ≥ k`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Lấp các khoảng trống bằng công thức

In [8]:
def min_k_sum_gaps(nums, k):
    total, prev = 0, 0
    for x in sorted(set(nums)):
        gap = x - prev - 1                        # số lượng số trống trong (prev, x)
        if gap > 0:
            take = min(gap, k)
            total += range_sum(prev + 1, prev + take)
            k -= take
            if k == 0:
                return total
        prev = x
    return total + range_sum(prev + 1, prev + k)  # phần còn lại sau phần tử lớn nhất

for nums_, k_, exp in examples:
    print(min_k_sum_gaps(nums_, k_), "(kỳ vọng", exp, ")")

5 (kỳ vọng 5 )
25 (kỳ vọng 25 )


**Độ phức tạp:** `O(n log n)` (sắp) + `O(n)`.

### 7.2 Cách 3 — Đẩy ngưỡng `m`

In [9]:
def min_k_sum_push(nums, k):
    m = k
    removed = 0
    for x in sorted(set(nums)):
        if x <= m:
            m += 1
            removed += x
        else:
            break
    return m * (m + 1) // 2 - removed

for nums_, k_, exp in examples:
    print(min_k_sum_push(nums_, k_), "(kỳ vọng", exp, ")")

5 (kỳ vọng 5 )
25 (kỳ vọng 25 )


**Vì sao "đẩy" đúng khi duyệt tăng dần?** Khi xét `x`, ngưỡng `m` hiện tại đã tính đúng cho các phần tử nhỏ hơn `x`. Nếu `x ≤ m`, `x` nằm trong `[1, m]` ⇒ chỉ còn `m − (số phần tử ≤ m)` số trống, thiếu một ⇒ `m + 1`. Số mới `m + 1` có thể trùng một phần tử chưa xét — vòng lặp sẽ xét nó ngay sau (vì sắp tăng). **Độ phức tạp:** `O(n log n)`.

### 7.3 Cách 4 — Tìm nhị phân ngưỡng + tổng tiền tố

In [10]:
def min_k_sum_bsearch(nums, k):
    u = sorted(set(nums))
    pre = [0]
    for v in u:
        pre.append(pre[-1] + v)
    lo, hi = 1, k + len(u)                        # m không vượt k + số phần tử phân biệt
    while lo < hi:
        mid = (lo + hi) // 2
        if mid - bisect_right(u, mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    m = lo
    c = bisect_right(u, m)
    return m * (m + 1) // 2 - pre[c]

for nums_, k_, exp in examples:
    print(min_k_sum_bsearch(nums_, k_), "(kỳ vọng", exp, ")")

5 (kỳ vọng 5 )
25 (kỳ vọng 25 )


**Độ phức tạp:** `O(n log n)` (sắp) + `O(log(k + n) · log n)`. Hữu ích khi cần trả lời **nhiều** truy vấn `k` trên cùng `nums` (bài tập 5).

### 7.4 Cách 5 — Duyệt mảng sắp với chỉ số (không cần ngưỡng)

Với `u` đã sắp, phần tử thứ `i` (0-based) là `u[i]`; số số thiếu **trước** `u[i]` là `u[i] − 1 − i`. Tìm chỉ số đầu tiên `i` mà số thiếu trước nó `≥ k` ⇒ số thiếu thứ `k` là `k + i`. Đây là công thức đóng cho ngưỡng `m = k + (số phần tử < m)`.

In [11]:
def min_k_sum_index(nums, k):
    u = sorted(set(nums))
    i = 0
    while i < len(u) and u[i] - 1 - i < k:        # số thiếu trước u[i] chưa đủ k
        i += 1
    m = k + i                                      # số thiếu thứ k
    return m * (m + 1) // 2 - sum(u[:i])

for nums_, k_, exp in examples:
    print(min_k_sum_index(nums_, k_), "(kỳ vọng", exp, ")")

5 (kỳ vọng 5 )
25 (kỳ vọng 25 )


**Vì sao `m = k + i`?** Có đúng `i` phần tử của `u` nhỏ hơn `m` (các `u[0..i−1]`, vì `u[i−1] − 1 − (i−1) < k` nghĩa là chúng nằm trước số thiếu thứ `k`), nên trong `[1, m]` có `m − i = k` số thiếu, và `m` chính là số thiếu thứ `k` (vì `u[i] − 1 − i ≥ k` ⇒ `u[i] > m`).

## 8. TRACE từng bước

### 8.1 Trace cách 3 (đẩy ngưỡng) trên hai ví dụ

In [12]:
def trace_push(nums, k):
    m = k; removed = 0
    print(f"  nums={nums}, k={k}: bắt đầu m = {m}")
    for x in sorted(set(nums)):
        if x <= m:
            m += 1; removed += x
            print(f"    x = {x} ≤ m -> chiếm chỗ, m = {m}, đã trừ {removed}")
        else:
            print(f"    x = {x} > m = {m} -> dừng"); break
    ans = m * (m + 1) // 2 - removed
    print(f"  => 1 + ... + {m} = {m * (m + 1) // 2}, trừ {removed} -> {ans}")
    return ans

for nums_, k_, _ in examples:
    trace_push(nums_, k_)

  nums=[1, 4, 25, 10, 25], k=2: bắt đầu m = 2
    x = 1 ≤ m -> chiếm chỗ, m = 3, đã trừ 1
    x = 4 > m = 3 -> dừng
  => 1 + ... + 3 = 6, trừ 1 -> 5
  nums=[5, 6], k=6: bắt đầu m = 6
    x = 5 ≤ m -> chiếm chỗ, m = 7, đã trừ 5
    x = 6 ≤ m -> chiếm chỗ, m = 8, đã trừ 11
  => 1 + ... + 8 = 36, trừ 11 -> 25


### 8.2 Trace cách 2 (lấp khoảng trống) trên dữ liệu tự dựng có trùng (đã đối chiếu vét cạn — đối chiếu lại đề gốc)

In [13]:
def trace_gaps(nums, k):
    total, prev = 0, 0
    for x in sorted(set(nums)):
        gap = x - prev - 1
        if gap > 0:
            take = min(gap, k)
            part = range_sum(prev + 1, prev + take)
            total += part; k -= take
            print(f"    khoảng ({prev}, {x}): {gap} số trống, lấy {take} số {prev + 1}..{prev + take}, cộng {part} -> {total}, còn k = {k}")
            if k == 0:
                return total
        else:
            print(f"    khoảng ({prev}, {x}): không có số trống")
        prev = x
    part = range_sum(prev + 1, prev + k)
    print(f"    sau {prev}: lấy {k} số {prev + 1}..{prev + k}, cộng {part}")
    return total + part

demo = [3, 3, 1, 8, 2, 10]
r = trace_gaps(demo, 6)
print("  =>", r, "| vét cạn:", min_k_sum_brute(demo, 6))

    khoảng (0, 1): không có số trống
    khoảng (1, 2): không có số trống
    khoảng (2, 3): không có số trống
    khoảng (3, 8): 4 số trống, lấy 4 số 4..7, cộng 22 -> 22, còn k = 2
    khoảng (8, 10): 1 số trống, lấy 1 số 9..9, cộng 9 -> 31, còn k = 1
    sau 10: lấy 1 số 11..11, cộng 11
  => 42 | vét cạn: 42


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def minimalKSum(self, nums: List[int], k: int) -> int:
        m = k                          # số thiếu thứ k (ngưỡng), giả sử ban đầu không bị chặn
        removed = 0                    # tổng các giá trị trong nums nằm trong [1, m]
        for x in sorted(set(nums)):    # khử trùng rồi sắp tăng
            if x > m:
                break
            m += 1
            removed += x
        return m * (m + 1) // 2 - removed

for nums_, k_, exp in examples:
    assert Solution().minimalKSum(nums_, k_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Tham lam** (mục 6): lấy `k` số thiếu nhỏ nhất `g₁ < … < g_k` là tối ưu.
2. **Đặc trưng ngưỡng:** `g_k = m` là số nhỏ nhất với `m − |{x ∈ U : x ≤ m}| = k` (`U = set(nums)`), và tập được chọn là `[1, m] \ U`. Tổng `= m(m+1)/2 − Σ_{x ∈ U, x ≤ m} x`.
3. **Đẩy ngưỡng tính đúng `m`:** bất biến sau khi xét các phần tử `u₀ < … < u_{j−1}` (đều `≤ m`): `m = k + j`. Khi gặp `u_j ≤ m` ⇒ trong `[1, m]` có `j + 1` phần tử của `U` ⇒ chỉ `k − 1` số trống ⇒ tăng `m`. Khi gặp `u_j > m` ⇒ `[1, m]` có đúng `k` số trống và `m ∉ U` (vì `m < u_j` và mọi phần tử nhỏ hơn đã xét đều `< m`… chính xác hơn: nếu `m ∈ U` thì nó sẽ là `u_j` hoặc đã được xét) ⇒ `m` là số thiếu thứ `k`. Các phần tử sau lớn hơn nữa nên không ảnh hưởng.
4. **Trùng lặp:** `set(nums)` đảm bảo mỗi giá trị chỉ "chiếm chỗ" một lần.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `u = |set(nums)|`.

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt từng số | `O(n + k + n)` | `O(n)` |
| 2. Lấp khoảng trống | `O(n log n)` | `O(u)` |
| 3. Đẩy ngưỡng | `O(n log n)` | `O(u)` |
| 4. Tìm nhị phân + tiền tố | `O(n log n + log(k+u) log u)` | `O(u)` |
| 5. Chỉ số trong mảng sắp | `O(n log n)` | `O(u)` |

In [15]:
rnd = random.Random(2195)
big = [rnd.randint(1, 10**9) for _ in range(10**5)]
dense = list(range(1, 10**5 + 1))
for label_, nums_, k_ in [("ngẫu nhiên", big, 10**8), ("dày 1..1e5", dense, 10**8), ("k nhỏ", big, 3)]:
    row = []
    for name, fn in [("gaps", min_k_sum_gaps), ("push", min_k_sum_push), ("bsearch", min_k_sum_bsearch), ("index", min_k_sum_index), ("Solution", Solution().minimalKSum)]:
        t0 = time.perf_counter(); r = fn(nums_, k_); row.append(f"{name}={(time.perf_counter() - t0) * 1000:.0f}ms")
    print(f"{label_:<11} -> {r}: " + ", ".join(row))
t0 = time.perf_counter(); min_k_sum_brute(big, 10**6); print(f"brute với k = 1e6: {(time.perf_counter() - t0) * 1000:.0f} ms (k = 1e8 sẽ ~100 lần chậm hơn)")

ngẫu nhiên  -> 5000499755351438: gaps=31ms, push=26ms, bsearch=32ms, index=24ms, Solution=24ms


dày 1..1e5  -> 5010000050000000: gaps=6ms, push=6ms, bsearch=6ms, index=9ms, Solution=6ms
k nhỏ       -> 6: gaps=23ms, push=23ms, bsearch=29ms, index=23ms, Solution=23ms


brute với k = 1e6: 62 ms (k = 1e8 sẽ ~100 lần chậm hơn)


## 12. Bộ test

In [16]:
ALL = {
    "gaps": min_k_sum_gaps,
    "push": min_k_sum_push,
    "bsearch": min_k_sum_bsearch,
    "index": min_k_sum_index,
    "Solution": lambda a, k: Solution().minimalKSum(a, k),
}

def check(nums, k, exp=None, brute=True):
    ref = min_k_sum_brute(nums, k) if brute else min_k_sum_gaps(nums, k)
    for name, fn in ALL.items():
        got = fn(list(nums), k)
        assert got == ref, (name, nums, k, got, ref)
    if exp is not None:
        assert ref == exp, (nums, k, ref, exp)
    return ref

for nums_, k_, exp in examples:
    check(nums_, k_, exp)
print("Ví dụ đề: OK")

check([1], 1, 2)                       # 1 có sẵn -> thêm 2
check([1, 2, 3], 2, 9)                 # 4 + 5
check([10**9], 1, 1)
check([2, 2, 2], 2, 4)                 # trùng: 1 + 3
check([1, 3, 5], 3, 2 + 4 + 6)
check(list(range(1, 11)), 1, 11)
check([10**9] * 3, 10**8, 10**8 * (10**8 + 1) // 2, brute=False)
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(3000):
    n = rnd.randint(1, 12)
    nums_ = [rnd.randint(1, 20) for _ in range(n)]
    check(nums_, rnd.randint(1, 25))
print("3000 bộ nhỏ (nhiều trùng, nhiều số nhỏ) so với vét cạn: OK")

for _ in range(10):
    n = rnd.randint(1, 10**5)
    nums_ = [rnd.randint(1, rnd.choice([10**3, 10**6, 10**9])) for _ in range(n)]
    check(nums_, rnd.randint(1, 10**8), brute=False)
print("10 bộ lớn (đối chiếu chéo): OK")

Ví dụ đề: OK
Edge case: OK
3000 bộ nhỏ (nhiều trùng, nhiều số nhỏ) so với vét cạn: OK


10 bộ lớn (đối chiếu chéo): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không khử trùng `nums`

In [17]:
def bug_no_dedup(nums, k):
    m = k; removed = 0
    for x in sorted(nums):                         # SAI: 25 xuất hiện hai lần -> đẩy m hai lần
        if x > m: break
        m += 1; removed += x
    return m * (m + 1) // 2 - removed

nums_, k_ = [2, 2, 2], 2
print("sai :", bug_no_dedup(nums_, k_), "| đúng:", Solution().minimalKSum(nums_, k_))

sai : 9 | đúng: 4


**Vì sao sai:** mỗi bản sao của `2` làm ngưỡng tăng thêm một, như thể có ba số khác nhau bị chiếm. **Sửa:** `set(nums)`.

### Lỗi 2 — Không sắp trước khi đẩy ngưỡng

In [18]:
def bug_unsorted_list(nums, k):
    m = k; removed = 0
    for x in dict.fromkeys(nums):                  # khử trùng nhưng GIỮ thứ tự đầu vào (không sắp)
        if x <= m:
            m += 1; removed += x
    return m * (m + 1) // 2 - removed

nums_, k_ = [5, 4], 4
print("sai :", bug_unsorted_list(nums_, k_), "| đúng:", Solution().minimalKSum(nums_, k_))

sai : 11 | đúng: 12


**Vì sao sai:** gặp `5` khi `m = 4` ⇒ bỏ qua; sau đó gặp `4` ⇒ `m = 5` — nhưng `5` đã bị bỏ qua nên được coi là số trống, trong khi nó có trong `nums`. Kết quả lấy `{1, 2, 3, 5}` thay vì `{1, 2, 3, 6}`. **Sửa:** `sorted(set(nums))` — duyệt tăng dần đảm bảo mọi phần tử `≤ m` (kể cả sau khi `m` tăng) đều được xét.

### Lỗi 3 — Lấp khoảng trống lệch một: đếm `x − prev` số trống thay vì `x − prev − 1`

In [19]:
def bug_gap_off_by_one(nums, k):
    total, prev = 0, 0
    for x in sorted(set(nums)):
        gap = x - prev                              # SAI: tính cả chính x
        take = min(gap, k)
        total += range_sum(prev + 1, prev + take); k -= take
        if k == 0: return total
        prev = x
    return total + range_sum(prev + 1, prev + k)

nums_, k_ = [5, 6], 6
print("sai :", bug_gap_off_by_one(nums_, k_), "| đúng:", Solution().minimalKSum(nums_, k_))

sai : 21 | đúng: 25


**Sửa:** số số trống giữa `prev` và `x` (không tính hai đầu) là `x − prev − 1`.

### Lỗi 4 — Tổng `1..k` rồi chỉ trừ các phần tử `≤ k` (quên bù phần bị đẩy)

In [20]:
def bug_no_push(nums, k):
    return k * (k + 1) // 2 - sum(x for x in set(nums) if x <= k)       # SAI: thiếu các số thay thế

nums_, k_ = [1, 4, 25, 10, 25], 2
print("sai :", bug_no_push(nums_, k_), "| đúng:", Solution().minimalKSum(nums_, k_))

sai : 2 | đúng: 5


**Vì sao sai:** bỏ `1` khỏi `{1, 2}` thì chỉ còn **một** số — cần thêm `3` để đủ `k = 2`. **Sửa:** đẩy ngưỡng.

### Lỗi 5 — Dùng phép chia số thực trong công thức tổng

In [21]:
def bug_float_formula(nums, k):
    m = k; removed = 0
    for x in sorted(set(nums)):
        if x > m: break
        m += 1; removed += x
    return int(m * (m + 1) / 2) - removed            # SAI khi m(m+1) vượt 2^53

nums_, k_ = [10**9] * 3, 10**8
print("trong ràng buộc đề (m ≤ ~1.001e8):", bug_float_formula(nums_, k_) == Solution().minimalKSum(nums_, k_))
big_m = 10**9 + 7                                  # ngoài ràng buộc: minh họa thói quen nguy hiểm
print("với m = 1e9 + 7:", int(big_m * (big_m + 1) / 2), "vs đúng", big_m * (big_m + 1) // 2)

trong ràng buộc đề (m ≤ ~1.001e8): True
với m = 1e9 + 7: 500000007500000000 vs đúng 500000007500000028


**Vì sao là lỗi:** với ràng buộc của đề (`m ≤ k + n ≈ 1.001·10⁸`), phép chia số thực **tình cờ** vẫn đúng (đã kiểm ở trên), nhưng chỉ cần `k` lớn hơn một chút (ví dụ `m ≈ 10⁹`) là `m(m+1)/2` vượt độ chính xác 53 bit và lệch. Không nên phụ thuộc may mắn. **Sửa:** `// 2` số nguyên.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn `k` số **khác nhau** nhỏ nhất thỏa điều kiện" ⇒ tham lam + **công thức đoạn** thay vì duyệt.
- `k` rất lớn (`10⁸`) ⇒ chắc chắn cần công thức `O(1)` cho mỗi đoạn.
- Luôn **khử trùng + sắp** trước khi xử lý "khoảng trống".
- Hàm "số phần tử thiếu ≤ m" không giảm ⇒ tìm nhị phân được (LeetCode 1539 — Kth Missing Positive Number).
- C++/Java: kết quả tới `~5·10¹⁵` — dùng `long long`/`long`.

## 15. Biến thể của bài

1. **LeetCode 1539 — Kth Missing Positive Number:** chỉ cần ngưỡng `m` (không cần tổng).
2. **LeetCode 41 — First Missing Positive** (`k = 1`, yêu cầu `O(1)` bộ nhớ).
3. **Tổng lớn nhất** của `k` số thiếu trong `[1, L]` — đối xứng, duyệt từ trên xuống.
4. **Nhiều truy vấn `k`** trên cùng `nums` — sắp một lần, tìm nhị phân mỗi truy vấn (bài tập 5).
5. **Chỉ được thêm số chẵn** (hoặc số trong một cấp số cộng) — công thức đoạn với bước 2.

In [22]:
def kth_missing_1539(arr, k):
    """arr tăng nghiêm ngặt; số dương thiếu thứ k."""
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] - mid - 1 < k: lo = mid + 1
        else: hi = mid
    return lo + k

def kth_missing_brute(arr, k):
    return first_missing(arr, k)[-1]

rnd_v = random.Random(15)
for _ in range(1000):
    arr = sorted(set(rnd_v.randint(1, 30) for _ in range(rnd_v.randint(1, 10))))
    k = rnd_v.randint(1, 20)
    assert kth_missing_1539(arr, k) == kth_missing_brute(arr, k)
print("1539 [2,3,4,7,11], k=5 ->", kth_missing_1539([2, 3, 4, 7, 11], 5), "| khớp vét cạn trên 1000 bộ")

1539 [2,3,4,7,11], k=5 -> 9 | khớp vét cạn trên 1000 bộ


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt từng số | vét cạn | `O(n + k)` | `O(n)` | kiểm thử, `k` nhỏ |
| 2. Lấp khoảng trống | tổng đoạn | `O(n log n)` | `O(u)` | trực quan |
| 3. Đẩy ngưỡng | `m = k + #chiếm` | `O(n log n)` | `O(u)` | **nộp bài** |
| 4. Tìm nhị phân | `missing(m) ≥ k` | `O(n log n)` + `O(log²)` | `O(u)` | nhiều truy vấn |
| 5. Chỉ số | `u[i] − 1 − i` | `O(n log n)` | `O(u)` | công thức gọn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minimalKSum([1, 2, 3, 4], 3)`?

**Bài 2 (tính tay):** `nums = [3, 3, 1, 8, 2, 10]`, `k = 6`: chạy cách 3 bằng tay.

**Bài 3 (điền TODO):** Hoàn thiện `kth_missing(nums, k)` — **số** thiếu thứ `k` (không cần tổng), `nums` bất kỳ (có trùng, chưa sắp).

**Bài 4 (tự viết lại):** Viết lại bài toán trả về **danh sách nén** các đoạn được thêm (dạng `[(a, b), …]`, mỗi đoạn là `a..b`), không liệt kê từng số; kiểm tổng và số lượng.

**Bài 5 (mini challenge):** Trả lời `q` truy vấn `k₁, k₂, …` trên cùng `nums` (`q` tới `10⁵`). Gợi ý: sắp + khử trùng một lần, tổng tiền tố, mỗi truy vấn tìm nhị phân chỉ số `i` (cách 5).

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def kth_missing(nums, k):
    # TODO: như đẩy ngưỡng, trả về m
    pass

# Bài 4 — TODO
def added_ranges(nums, k):
    # TODO: như cách 2 nhưng ghi (prev + 1, prev + take)
    pass

# Bài 5 — TODO
def answer_queries(nums, ks):
    # TODO: u = sorted(set(nums)), pre; với mỗi k: tìm i đầu tiên có u[i] - 1 - i >= k
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `1..4` đều có.
- Mức 2: Lấy `5, 6, 7`.
- Mức 3: `18`.

**Bài 2**
- Mức 1: `u = [1, 2, 3, 8, 10]`, bắt đầu `m = 6`.
- Mức 2: `1, 2, 3` đều `≤ m` ⇒ `m = 9`; `8 ≤ 9` ⇒ `m = 10`; `10 ≤ 10` ⇒ `m = 11`.
- Mức 3: `66 − (1+2+3+8+10) = 42`.

**Bài 3**
- Mức 1: Trả `m` của cách 3.
- Mức 2: Khử trùng + sắp.
- Mức 3: So với `first_missing(nums, k)[-1]`.

**Bài 4**
- Mức 1: Mỗi khoảng trống đóng góp tối đa một đoạn.
- Mức 2: Đoạn cuối `(prev + 1, prev + k)`.
- Mức 3: Kiểm `Σ (b − a + 1) = k` và `Σ range_sum(a, b) = ` đáp án.

**Bài 5**
- Mức 1: `missing_before[i] = u[i] − 1 − i` không giảm.
- Mức 2: `i = bisect_left(missing_before, k)`.
- Mức 3: `m = k + i`, đáp án `m(m+1)/2 − pre[i]`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
from bisect import bisect_left

# Bài 1
r1 = Solution().minimalKSum([1, 2, 3, 4], 3)
print("Bài 1:", r1); assert r1 == 18

# Bài 2
print("Bài 2:"); r2 = trace_push([3, 3, 1, 8, 2, 10], 6); assert r2 == 42 == min_k_sum_brute([3, 3, 1, 8, 2, 10], 6)

# Bài 3
def kth_missing_sol(nums, k):
    m = k
    for x in sorted(set(nums)):
        if x > m: break
        m += 1
    return m

rnd = random.Random(3)
for _ in range(2000):
    a = [rnd.randint(1, 20) for _ in range(rnd.randint(1, 10))]; k = rnd.randint(1, 20)
    assert kth_missing_sol(a, k) == first_missing(a, k)[-1]
print("Bài 3 OK, ví dụ:", kth_missing_sol([5, 6], 6), kth_missing_sol([2, 2, 2], 2))

Bài 1: 18
Bài 2:
  nums=[3, 3, 1, 8, 2, 10], k=6: bắt đầu m = 6
    x = 1 ≤ m -> chiếm chỗ, m = 7, đã trừ 1
    x = 2 ≤ m -> chiếm chỗ, m = 8, đã trừ 3
    x = 3 ≤ m -> chiếm chỗ, m = 9, đã trừ 6
    x = 8 ≤ m -> chiếm chỗ, m = 10, đã trừ 14
    x = 10 ≤ m -> chiếm chỗ, m = 11, đã trừ 24
  => 1 + ... + 11 = 66, trừ 24 -> 42
Bài 3 OK, ví dụ: 8 3


In [25]:
# Bài 4
def added_ranges_sol(nums, k):
    out, prev = [], 0
    for x in sorted(set(nums)):
        gap = x - prev - 1
        if gap > 0:
            take = min(gap, k); out.append((prev + 1, prev + take)); k -= take
            if k == 0: return out
        prev = x
    out.append((prev + 1, prev + k))
    return out

# Bài 5
def answer_queries_sol(nums, ks):
    u = sorted(set(nums))
    before = [u[i] - 1 - i for i in range(len(u))]       # số thiếu trước u[i]
    pre = [0]
    for v in u: pre.append(pre[-1] + v)
    res = []
    for k in ks:
        i = bisect_left(before, k)
        m = k + i
        res.append(m * (m + 1) // 2 - pre[i])
    return res

rnd = random.Random(5)
for _ in range(1000):
    a = [rnd.randint(1, 30) for _ in range(rnd.randint(1, 10))]; k = rnd.randint(1, 25)
    rg = added_ranges_sol(a, k)
    assert sum(b - a_ + 1 for a_, b in rg) == k and sum(range_sum(a_, b) for a_, b in rg) == Solution().minimalKSum(a, k)
    assert all(not (a_ <= x <= b) for a_, b in rg for x in a)
    ks = [rnd.randint(1, 40) for _ in range(5)]
    assert answer_queries_sol(a, ks) == [Solution().minimalKSum(a, kk) for kk in ks]
print("Bài 4, Bài 5 OK; ví dụ đoạn:", added_ranges_sol([5, 6], 6), added_ranges_sol([1, 4, 25, 10, 25], 2))

Bài 4, Bài 5 OK; ví dụ đoạn: [(1, 4), (7, 8)] [(2, 3)]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đẩy ngưỡng":**

```python
m, removed = k, 0
for x in sorted(set(nums)):
    if x > m: break
    m += 1; removed += x
return m * (m + 1) // 2 - removed
```

**Checklist:**
- [ ] Khử trùng (`set`) **và** sắp tăng?
- [ ] Đẩy ngưỡng khi `x ≤ m` (so với `m` hiện tại, không phải `k` ban đầu)?
- [ ] Khoảng trống `x − prev − 1`?
- [ ] Công thức số nguyên `// 2`?
- [ ] C++/Java: 64 bit?
- [ ] Chữ ký đúng: `def minimalKSum(self, nums: List[int], k: int) -> int`?